# S03 · Parámetros de generación con DeepSeek

Laboratorio opcional para comparar `temperature`, `top_p` y `max_tokens` usando la API OpenAI-compatible de DeepSeek.

🔐 No pegues la key en una celda ni subas `.env`: usa Colab Secrets con `DEEPSEEK_API_KEY`.


In [ ]:
!pip -q install openai


In [ ]:
import os
from getpass import getpass
from openai import OpenAI
try:
    from google.colab import userdata
    api_key = userdata.get('DEEPSEEK_API_KEY')
except Exception:
    api_key = None
if not api_key:
    api_key = getpass('DEEPSEEK_API_KEY (entrada oculta): ')
assert api_key and api_key.startswith('sk-'), 'No se recibió una key válida.'
client = OpenAI(api_key=api_key, base_url='https://api.deepseek.com')
MODELO = 'deepseek-chat'
print('✅ Cliente listo. La key no se muestra ni se guarda en el notebook.')


## 1 · Comparación controlada

DeepSeek permite configurar explícitamente estos parámetros.


In [ ]:
pregunta = '''Propón exactamente 5 nombres distintos para una aplicación que ayuda a organizar tareas personales.
Devuelve una lista numerada. Cada nombre debe ser diferente de los demás, no uses Nimbus y añade una breve explicación de cada propuesta.'''
configuraciones = [
    {'nombre':'estable', 'temperature':0.0, 'top_p':0.9, 'max_tokens':240},
    {'nombre':'balanceada', 'temperature':0.5, 'top_p':0.9, 'max_tokens':240},
    {'nombre':'creativa', 'temperature':0.9, 'top_p':0.95, 'max_tokens':240},
]
resultados = []
for cfg in configuraciones:
    r = client.chat.completions.create(model=MODELO, messages=[{'role':'user','content':pregunta}],
        temperature=cfg['temperature'], top_p=cfg['top_p'], max_tokens=cfg['max_tokens'])
    resultados.append({**cfg, 'respuesta':r.choices[0].message.content,
        'finish_reason':r.choices[0].finish_reason, 'prompt_tokens':r.usage.prompt_tokens,
        'completion_tokens':r.usage.completion_tokens})
for x in resultados:
    print(f"\n--- {x['nombre']} · temp={x['temperature']} · top_p={x['top_p']} · max_tokens={x['max_tokens']}")
    print(x['respuesta'])
    print('tokens:', x['prompt_tokens'], '→', x['completion_tokens'], '| finish:', x['finish_reason'])


## 2 · Interpretación

Compara cómo cambia la creatividad, la variedad de nombres y la longitud de las explicaciones. En este ejercicio la temperatura sí es relevante porque estamos pidiendo ideación, no una respuesta factual. Luego volveremos a Neptuno para demostrar que subir la temperatura no reemplaza la evidencia. Comparte únicamente el `.ipynb`; cada alumno configura su propia key en Colab Secrets.
